
# Spatiotemporal Deep Learning for Heatwave Prediction over India

## ERA5 Historical Data Extraction and Dataset Preparation

This notebook documents the extraction, processing and merging of ERA5 reanalysis data for the heatwave prediction project.

The dataset covers the Indian geographical region and contains daily atmospheric variables derived from six-hourly ERA5 observations.

### Project Details
- Data sources: WeatherBench 2 and Copernicus Climate Data Store (CDS)
- Geographic region: India
- Spatial resolution: 1.5° × 1.5°
- Historical period: 1990–2023
- Additional data: Selected periods from 2023–2026
- Output format: NetCDF (.nc)

### Main Processing Stages
1. Environment setup and Google Drive connection
2. Dataset configuration and daily aggregation functions
3. Historical ERA5 data extraction
4. Recent ERA5 data retrieval through CDS
5. Dataset inspection and merging
6. Final dataset validation



## 1. Environment Setup and Google Drive Connection

The required Python libraries are installed to support climate data retrieval, NetCDF processing, cloud-based dataset access and data analysis.

Google Drive is mounted to store the downloaded and processed datasets.


In [4]:
!pip install -q xarray zarr gcsfs dask netCDF4 cdsapi

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

folder = "/content/drive/MyDrive/heatwave_project/"
for name in sorted(os.listdir(folder)):
    print(name)

hist_1990.nc
hist_1991.nc
hist_1992.nc
hist_1993.nc
hist_1994.nc
hist_1995.nc
hist_1996.nc
hist_1997.nc
hist_1998.nc
hist_1999.nc
hist_2000.nc
hist_2001.nc
hist_2002.nc
hist_2003.nc
hist_2004.nc
hist_2005.nc
hist_2006.nc
hist_2007.nc
hist_2008.nc
hist_2009.nc
hist_2010.nc
hist_2011.nc
hist_2012.nc
hist_2013.nc
hist_2014.nc
hist_2015.nc
hist_2016.nc
hist_2017.nc
hist_2018.nc
hist_2019.nc
hist_2020.nc
hist_2021.nc
hist_2022.nc
hist_2023.nc
india_daily_1990_2023.csv
india_daily_1990_2023.nc
sl_2023.nc



## 2. Dataset Configuration and Daily Aggregation

This section defines the geographical boundaries, dataset location and WeatherBench 2 source.

It also contains the functions used to convert six-hourly ERA5 observations into daily atmospheric variables.

The following variables are generated:

- tmax: Daily maximum 2 m temperature
- tmean: Daily mean 2 m temperature
- u10: Daily mean eastward wind component
- v10: Daily mean northward wind component
- msl: Daily mean sea-level pressure
- z500: Daily mean geopotential at 500 hPa
- q850: Daily mean specific humidity at 850 hPa
- t850: Daily mean temperature at 850 hPa


In [5]:
import os
import getpass
import xarray as xr
from dataclasses import dataclass


@dataclass
class Config:
    """All project settings in one place."""
    folder: str = "/content/drive/MyDrive/heatwave_project/"
    lat_min: float = 6.0
    lat_max: float = 37.5
    lon_min: float = 69.0
    lon_max: float = 97.5
    wb2_url: str = ("gs://weatherbench2/datasets/era5/"
                    "1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")


cfg = Config()
os.makedirs(cfg.folder, exist_ok=True)


def daily_mean(data):
    """Average the 4 readings of each day into 1 value."""
    return data.resample(time="1D").mean()


def make_daily(t2m, u10, v10, msl, z500, q850, t850):
    """Turn 6-hourly maps into daily maps (temperatures in Celsius)."""
    return xr.Dataset({
        "tmax":  t2m.resample(time="1D").max() - 273.15,
        "tmean": daily_mean(t2m) - 273.15,
        "u10":   daily_mean(u10),
        "v10":   daily_mean(v10),
        "msl":   daily_mean(msl),
        "z500":  daily_mean(z500),
        "q850":  daily_mean(q850),
        "t850":  daily_mean(t850) - 273.15,
    })


## 3. Historical ERA5 Data Extraction (1990–2023)

Historical ERA5 data are obtained from the publicly available WeatherBench 2 dataset hosted on Google Cloud Storage.

The extraction process:

1. Selects the geographical region covering India.
2. Processes the data year by year to manage memory usage.
3. Extracts the required surface and pressure-level variables.
4. Converts six-hourly observations into daily atmospheric variables.
5. Converts temperature measurements from Kelvin to Celsius.
6. Saves the processed historical dataset in NetCDF format.

Output file: `india_daily_1990_2023.nc`


In [ ]:
import time
import dask

dask.config.set(scheduler="threads", num_workers=32)


class HistoricalData:
    """Loads 1990-2023 from the public WeatherBench2 copy of ERA5, one year at a time."""

    def __init__(self, config, first_year=1990, last_year=2023):
        self.cfg = config
        self.years = range(first_year, last_year + 1)
        self.path = config.folder + "india_daily_1990_2023.nc"
        self.ds = xr.open_zarr(config.wb2_url, storage_options={"token": "anon"})

    def _india(self, data, year):
        """Keep only the India region and the given year."""
        return data.sel(time=slice(f"{year}-01-01", f"{year}-12-31"),
                        latitude=slice(self.cfg.lat_min, self.cfg.lat_max),
                        longitude=slice(self.cfg.lon_min, self.cfg.lon_max))

    def _level(self, name, level, year):
        """Pick one pressure level of a variable."""
        data = self.ds[name].sel(level=level).drop_vars("level")
        return self._india(data, year)

    def build_year(self, year):
        """Make the daily maps of one year and save them (skip if already saved)."""
        path = self.cfg.folder + f"hist_{year}.nc"
        if os.path.exists(path):
            return path
        daily = make_daily(
            t2m=self._india(self.ds["2m_temperature"], year),
            u10=self._india(self.ds["10m_u_component_of_wind"], year),
            v10=self._india(self.ds["10m_v_component_of_wind"], year),
            msl=self._india(self.ds["mean_sea_level_pressure"], year),
            z500=self._level("geopotential", 500, year),
            q850=self._level("specific_humidity", 850, year),
            t850=self._level("temperature", 850, year),
        ).compute()
        daily.to_netcdf(path)
        return path

    def build(self):
        if os.path.exists(self.path):
            print("Final file already exists, skipping.")
            return xr.open_dataset(self.path)
        start = time.time()
        paths = []
        for year in self.years:
            paths.append(self.build_year(year))
            print(f"done {year}  ({(time.time() - start) / 60:.1f} min so far)")
        full = xr.concat([xr.open_dataset(p) for p in paths], dim="time")
        full.to_netcdf(self.path)
        return full


historical = HistoricalData(cfg).build()
print(historical)

done 1990  (0.7 min so far)
done 1991  (1.1 min so far)
done 1992  (1.5 min so far)
done 1993  (2.0 min so far)
done 1994  (2.5 min so far)
done 1995  (3.0 min so far)
done 1996  (3.5 min so far)
done 1997  (3.9 min so far)
done 1998  (4.5 min so far)
done 1999  (4.9 min so far)
done 2000  (5.4 min so far)
done 2001  (5.8 min so far)
done 2002  (6.4 min so far)
done 2003  (6.8 min so far)
done 2004  (7.3 min so far)
done 2005  (7.7 min so far)
done 2006  (8.3 min so far)
done 2007  (8.7 min so far)
done 2008  (9.2 min so far)
done 2009  (9.6 min so far)
done 2010  (10.2 min so far)
done 2011  (10.7 min so far)
done 2012  (11.1 min so far)
done 2013  (11.6 min so far)
done 2014  (12.1 min so far)
done 2015  (12.6 min so far)
done 2016  (13.0 min so far)
done 2017  (13.4 min so far)
done 2018  (14.0 min so far)
done 2019  (14.4 min so far)
done 2020  (14.8 min so far)
done 2021  (15.3 min so far)
done 2022  (15.8 min so far)
done 2023  (15.9 min so far)
<xarray.Dataset> Size: 162MB
Dimen


## 4. Recent ERA5 Data Retrieval Using Copernicus CDS

Additional ERA5 data are retrieved through the official Copernicus Climate Data Store API.

The requested variables are divided into:

- Single-level variables: 2 m temperature, 10 m wind components and mean sea-level pressure.
- Pressure-level variables: Geopotential, specific humidity and temperature at selected pressure levels.

The retrieval code includes request submission, queue monitoring, download management and storage of downloaded NetCDF files in Google Drive.

The following cells document the initial batch-based retrieval implementation.


In [ ]:
import glob
import json
import time
from ecmwf.datastores import Client

SINGLE_VARS = ["2m_temperature", "10m_u_component_of_wind",
               "10m_v_component_of_wind", "mean_sea_level_pressure"]
PRESSURE_VARS = ["geopotential", "specific_humidity", "temperature"]


def clean_cds(ds):
    """Make a Copernicus file look like the WeatherBench2 one."""
    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    if "expver" in ds.dims:
        ds = ds.sel(expver=1, drop=True).combine_first(ds.sel(expver=5, drop=True))
    ds = ds.drop_vars([n for n in ["expver", "number"] if n in ds.variables])
    if "pressure_level" in ds.dims:
        ds = ds.rename({"pressure_level": "level"})
    return ds


class CopernicusBatch:
    """Submits ALL Copernicus requests at once, then downloads each one when it is ready."""

    def __init__(self, config, years=(2023, 2024, 2025, 2026)):
        self.cfg = config
        self.years = years
        self.area = [37.5, 69.0, 7.5, 97.5]   # North, West, South, East
        self.ids_path = config.folder + "request_ids.json"
        self.ids = {}
        if os.path.exists(self.ids_path):
            with open(self.ids_path) as f:
                self.ids = json.load(f)
        key = getpass.getpass("Paste your CDS API key and press Enter: ")
        self.client = Client(url="https://cds.climate.copernicus.eu/api", key=key)

    def _path(self, name):
        return self.cfg.folder + name + ".nc"

    def _save_ids(self):
        with open(self.ids_path, "w") as f:
            json.dump(self.ids, f)

    def _request(self, year, months, variables, levels=None):
        """Build one request for some months of one year."""
        request = dict(
            product_type=["reanalysis"],
            variable=variables,
            year=[str(year)],
            month=[f"{m:02d}" for m in months],
            day=[f"{d:02d}" for d in range(1, 32)],
            time=["00:00", "06:00", "12:00", "18:00"],
            data_format="netcdf",
            download_format="unarchived",
            area=self.area,
            grid=[1.5, 1.5],
        )
        if levels:
            request["pressure_level"] = levels
        return request

    def make_jobs(self):
        """List of every request we need: file name -> (dataset, request)."""
        jobs = {}
        for year in self.years:
            last_month = 10 if year == 2026 else 12
            months = list(range(1, last_month + 1))
            jobs[f"sl_{year}"] = ("reanalysis-era5-single-levels",
                                  self._request(year, months, SINGLE_VARS))
            for i in range(0, len(months), 3):
                part = months[i:i + 3]
                name = f"pl_{year}_{part[0]:02d}_{part[-1]:02d}"
                jobs[name] = ("reanalysis-era5-pressure-levels",
                              self._request(year, part, PRESSURE_VARS, levels=["500", "850"]))
        return jobs

    def submit_all(self, jobs):
        """Send every request that is not saved and not already submitted."""
        for name, (dataset, request) in jobs.items():
            if os.path.exists(self._path(name)) or name in self.ids:
                continue
            try:
                remote = self.client.submit(dataset, request)
                self.ids[name] = remote.request_id
                self._save_ids()
                print("submitted", name)
            except Exception as error:
                print("could not submit", name, "- will retry:", str(error)[:100])

    def download_ready(self):
        """Download every request that has finished."""
        for name, request_id in list(self.ids.items()):
            if os.path.exists(self._path(name)):
                continue
            try:
                remote = self.client.get_remote(request_id)
                status = remote.status
                if status == "successful":
                    remote.download(self._path(name))
                    print("downloaded", name)
                elif status not in ("accepted", "running"):
                    raise RuntimeError("status " + status)
            except Exception as error:
                print("request", name, "failed, submitting again:", str(error)[:80])
                del self.ids[name]
                self._save_ids()

    def run(self):
        """Submit everything, then check every 2 minutes until all files are saved."""
        jobs = self.make_jobs()
        while True:
            self.submit_all(jobs)
            self.download_ready()
            saved = sum(os.path.exists(self._path(n)) for n in jobs)
            print(f"{time.strftime('%H:%M')}  {saved} of {len(jobs)} files saved")
            if saved == len(jobs):
                break
            time.sleep(120)

    def _open_all(self, prefix):
        paths = []
        for year in self.years:
            paths += sorted(glob.glob(self.cfg.folder + f"{prefix}_{year}*.nc"))
        files = [clean_cds(xr.open_dataset(p)) for p in paths]
        return xr.concat(files, dim="time").sortby("time")

    def build_daily(self):
        """Turn the saved files into daily maps and save them as one file."""
        sl = self._open_all("sl")
        pl = self._open_all("pl")
        daily = make_daily(
            t2m=sl["t2m"], u10=sl["u10"], v10=sl["v10"], msl=sl["msl"],
            z500=pl["z"].sel(level=500).drop_vars("level"),
            q850=pl["q"].sel(level=850).drop_vars("level"),
            t850=pl["t"].sel(level=850).drop_vars("level"),
        ).compute()
        daily.to_netcdf(self.cfg.folder + "india_daily_2023_2026.nc")
        return daily


batch = CopernicusBatch(cfg)
batch.run()
recent = batch.build_daily()
print(recent)

Paste your CDS API key and press Enter: ··········
submitted pl_2023_01_03
submitted pl_2023_04_06
submitted pl_2023_07_09
submitted pl_2023_10_12
submitted sl_2024
submitted pl_2024_01_03
submitted pl_2024_04_06
submitted pl_2024_07_09


submitted pl_2024_10_12
submitted sl_2025
submitted pl_2025_01_03
submitted pl_2025_04_06
submitted pl_2025_07_09
submitted pl_2025_10_12
submitted sl_2026
submitted pl_2026_01_03
submitted pl_2026_04_06
submitted pl_2026_07_09
submitted pl_2026_10_10


2988ff4002fd6bafe0fad6ba5960f7f3.nc:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

downloaded pl_2023_01_03
request pl_2023_04_06 failed, submitting again: status rejected
request pl_2023_07_09 failed, submitting again: status rejected
request pl_2023_10_12 failed, submitting again: status rejected
request sl_2024 failed, submitting again: status rejected
request pl_2024_01_03 failed, submitting again: status rejected
request pl_2024_04_06 failed, submitting again: status rejected
request pl_2024_07_09 failed, submitting again: status rejected
request pl_2026_04_06 failed, submitting again: status rejected
request pl_2026_07_09 failed, submitting again: status rejected
request pl_2026_10_10 failed, submitting again: status rejected
03:48  2 of 20 files saved


KeyboardInterrupt: 


### 4.1 Queue-Managed Retrieval Approach

A queue-managed retrieval approach is implemented to limit the number of active CDS requests.

This approach checks the status of submitted requests, downloads completed files and handles rejected requests through retry logic.


In [ ]:
import glob
import json
import time
from ecmwf.datastores import Client

SINGLE_VARS = ["2m_temperature", "10m_u_component_of_wind",
               "10m_v_component_of_wind", "mean_sea_level_pressure"]
PRESSURE_VARS = ["geopotential", "specific_humidity", "temperature"]


def clean_cds(ds):
    """Make a Copernicus file look like the WeatherBench2 one."""
    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    if "expver" in ds.dims:
        ds = ds.sel(expver=1, drop=True).combine_first(ds.sel(expver=5, drop=True))
    ds = ds.drop_vars([n for n in ["expver", "number"] if n in ds.variables])
    if "pressure_level" in ds.dims:
        ds = ds.rename({"pressure_level": "level"})
    return ds


class CopernicusQueue:
    """Downloads Copernicus data politely: only a few requests wait at a time."""

    DATASETS = {"sl": "reanalysis-era5-single-levels",
                "pl": "reanalysis-era5-pressure-levels"}

    def __init__(self, config, years=(2023, 2024, 2025, 2026),
                 months_per_request=6, max_active=2):
        self.cfg = config
        self.years = years
        self.max_active = {"sl": max_active, "pl": max_active}
        self.area = [37.5, 69.0, 7.5, 97.5]   # North, West, South, East
        self.ids_path = config.folder + "request_ids.json"
        self.ids = {}                           # file name -> request id
        if os.path.exists(self.ids_path):
            with open(self.ids_path) as f:
                self.ids = json.load(f)
        self.jobs = self._make_jobs(months_per_request)
        self.pause_until = 0
        key = getpass.getpass("Paste your CDS API key and press Enter: ")
        self.client = Client(url="https://cds.climate.copernicus.eu/api", key=key)

    # ---------- planning ----------
    def _make_jobs(self, months_per_request):
        """List every file we need: file name -> what to ask for."""
        jobs = {}
        for year in self.years:
            last_month = 10 if year == 2026 else 12
            months = list(range(1, last_month + 1))
            jobs[f"sl_{year}"] = dict(kind="sl", year=year, months=months)
            for i in range(0, len(months), months_per_request):
                part = months[i:i + months_per_request]
                name = f"pl_{year}_{part[0]:02d}_{part[-1]:02d}"
                jobs[name] = dict(kind="pl", year=year, months=part)
        return jobs

    def _split(self, name):
        """Replace one job by two jobs that each cover half of the months."""
        job = self.jobs.pop(name)
        half = len(job["months"]) // 2
        for part in (job["months"][:half], job["months"][half:]):
            new = f"{job['kind']}_{job['year']}_{part[0]:02d}_{part[-1]:02d}"
            self.jobs[new] = dict(kind=job["kind"], year=job["year"], months=part)

    def _path(self, name):
        return self.cfg.folder + name + ".nc"

    def _save_ids(self):
        with open(self.ids_path, "w") as f:
            json.dump(self.ids, f)

    def _request(self, job):
        request = dict(
            product_type=["reanalysis"],
            year=[str(job["year"])],
            month=[f"{m:02d}" for m in job["months"]],
            day=[f"{d:02d}" for d in range(1, 32)],
            time=["00:00", "06:00", "12:00", "18:00"],
            data_format="netcdf",
            download_format="unarchived",
            area=self.area,
            grid=[1.5, 1.5],
        )
        if job["kind"] == "sl":
            request["variable"] = SINGLE_VARS
        else:
            request["variable"] = PRESSURE_VARS
            request["pressure_level"] = ["500", "850"]
        return request

    # ---------- talking to Copernicus ----------
    def _submit(self, name):
        job = self.jobs[name]
        try:
            remote = self.client.submit(self.DATASETS[job["kind"]], self._request(job))
        except Exception as error:
            text = str(error)
            if ("cost limits" in text or "too large" in text) and len(job["months"]) > 1:
                print("request too large, splitting it in two:", name)
                self._split(name)
            else:
                print("could not submit", name, "- waiting 10 minutes:", text[:100])
                self.pause_until = time.time() + 600
            return
        self.ids[name] = remote.request_id
        self._save_ids()
        print("submitted", name)

    def submit_pending(self):
        """Send new requests, but keep only a few waiting per dataset."""
        if time.time() < self.pause_until:
            return
        for name in list(self.jobs):
            job = self.jobs.get(name)
            if job is None or name in self.ids or os.path.exists(self._path(name)):
                continue
            waiting = sum(1 for n in self.ids
                          if n in self.jobs and self.jobs[n]["kind"] == job["kind"]
                          and not os.path.exists(self._path(n)))
            if waiting >= self.max_active[job["kind"]]:
                continue
            self._submit(name)
            if time.time() < self.pause_until:
                return

    def check_requests(self):
        """Download finished requests. If one was rejected, slow down."""
        for name, request_id in list(self.ids.items()):
            if os.path.exists(self._path(name)):
                continue
            try:
                remote = self.client.get_remote(request_id)
                status = remote.status
                if status == "successful":
                    remote.download(self._path(name))
                    print("downloaded", name)
                    continue
            except Exception as error:
                print("could not check", name, "-", str(error)[:80])
                continue
            if status not in ("accepted", "running"):
                kind = self.jobs.get(name, {}).get("kind", "pl")
                self.max_active[kind] = 1
                self.pause_until = time.time() + 600
                del self.ids[name]
                self._save_ids()
                print(f"{name} was rejected ({status}). Waiting 10 minutes, "
                      "then trying again with 1 request at a time.")

    def run(self):
        """Repeat every 2 minutes until every file is saved in Drive."""
        while True:
            self.submit_pending()
            self.check_requests()
            saved = sum(os.path.exists(self._path(n)) for n in self.jobs)
            waiting = sum(1 for n in self.ids if not os.path.exists(self._path(n)))
            print(f"{time.strftime('%H:%M')}  {saved} of {len(self.jobs)} files saved, "
                  f"{waiting} waiting at Copernicus")
            if saved == len(self.jobs):
                break
            time.sleep(120)

    # ---------- making the daily file ----------
    def _open_all(self, prefix):
        paths = []
        for year in self.years:
            paths += sorted(glob.glob(self.cfg.folder + f"{prefix}_{year}*.nc"))
        files = [clean_cds(xr.open_dataset(p)) for p in paths]
        return xr.concat(files, dim="time").sortby("time")

    def build_daily(self):
        sl = self._open_all("sl")
        pl = self._open_all("pl")
        daily = make_daily(
            t2m=sl["t2m"], u10=sl["u10"], v10=sl["v10"], msl=sl["msl"],
            z500=pl["z"].sel(level=500).drop_vars("level"),
            q850=pl["q"].sel(level=850).drop_vars("level"),
            t850=pl["t"].sel(level=850).drop_vars("level"),
        ).compute()
        daily.to_netcdf(self.cfg.folder + "india_daily_2023_2026.nc")
        return daily


queue = CopernicusQueue(cfg)
queue.run()
recent = queue.build_daily()
print(recent)

Paste your CDS API key and press Enter: ··········
submitted pl_2023_01_06
submitted pl_2023_07_12
could not check pl_2024_10_12 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check sl_2025 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check pl_2025_01_03 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check pl_2025_04_06 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check pl_2025_07_09 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check pl_2025_10_12 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check sl_2026 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
could not check pl_2026_01_03 - 404 Client Error: Not Found for url: https://cds.climate.copernicus.eu/api/retri
03:51  1 

KeyboardInterrupt: 

### 4.2 Request Status Handling

This section extends the queue-management procedure to handle requests that are no longer available on the CDS server.

It removes invalid request identifiers and allows the retrieval process to retry the affected requests.


In [ ]:
def check_requests(self):
    """Download finished requests. Drop requests that no longer exist."""
    for name, request_id in list(self.ids.items()):
        if os.path.exists(self._path(name)):
            continue
        try:
            remote = self.client.get_remote(request_id)
            status = remote.status
            if status == "successful":
                remote.download(self._path(name))
                print("downloaded", name)
                continue
        except Exception as error:
            if "404" in str(error) or "Not Found" in str(error):
                del self.ids[name]
                self._save_ids()
                print("old request", name, "no longer exists, removed from the list")
            else:
                print("could not check", name, "-", str(error)[:80])
            continue
        if status not in ("accepted", "running"):
            kind = self.jobs.get(name, {}).get("kind", "pl")
            self.max_active[kind] = 1
            self.pause_until = time.time() + 600
            del self.ids[name]
            self._save_ids()
            print(f"{name} was rejected ({status}). Waiting 10 minutes, "
                  "then trying again with 1 request at a time.")


CopernicusQueue.check_requests = check_requests

In [ ]:
queue.run()
recent = queue.build_daily()
print(recent)

old request pl_2024_10_12 no longer exists, removed from the list
old request sl_2025 no longer exists, removed from the list
old request pl_2025_01_03 no longer exists, removed from the list
old request pl_2025_04_06 no longer exists, removed from the list
old request pl_2025_07_09 no longer exists, removed from the list
old request pl_2025_10_12 no longer exists, removed from the list
old request sl_2026 no longer exists, removed from the list
old request pl_2026_01_03 no longer exists, removed from the list
03:56  1 of 12 files saved, 2 waiting at Copernicus
submitted sl_2024
submitted sl_2025
03:58  1 of 12 files saved, 4 waiting at Copernicus
sl_2024 was rejected (rejected). Waiting 10 minutes, then trying again with 1 request at a time.
sl_2025 was rejected (rejected). Waiting 10 minutes, then trying again with 1 request at a time.
04:00  1 of 12 files saved, 2 waiting at Copernicus
04:02  1 of 12 files saved, 2 waiting at Copernicus
04:04  1 of 12 files saved, 2 waiting at Coper

6df66d0876039a71213ec3a007f37343.nc:   0%|          | 0.00/3.63M [00:00<?, ?B/s]

downloaded pl_2023_01_06
06:25  2 of 12 files saved, 2 waiting at Copernicus
submitted pl_2024_01_06
06:27  2 of 12 files saved, 3 waiting at Copernicus
06:29  2 of 12 files saved, 3 waiting at Copernicus
06:31  2 of 12 files saved, 3 waiting at Copernicus
06:33  2 of 12 files saved, 3 waiting at Copernicus
06:35  2 of 12 files saved, 3 waiting at Copernicus
06:37  2 of 12 files saved, 3 waiting at Copernicus
06:39  2 of 12 files saved, 3 waiting at Copernicus
06:41  2 of 12 files saved, 3 waiting at Copernicus
06:43  2 of 12 files saved, 3 waiting at Copernicus
06:45  2 of 12 files saved, 3 waiting at Copernicus
06:47  2 of 12 files saved, 3 waiting at Copernicus
06:49  2 of 12 files saved, 3 waiting at Copernicus
06:51  2 of 12 files saved, 3 waiting at Copernicus
06:53  2 of 12 files saved, 3 waiting at Copernicus


### 4.3 Retrieval of Missing Dataset Periods

This section contains the additional retrieval procedure used to obtain missing ERA5 data periods.

The retrieval process uses the CDS API and stores completed downloads in the Google Drive project directory.


In [ ]:
import glob
import json
import time
from ecmwf.datastores import Client

SINGLE_VARS = ["2m_temperature", "10m_u_component_of_wind",
               "10m_v_component_of_wind", "mean_sea_level_pressure"]
PRESSURE_VARS = ["geopotential", "specific_humidity", "temperature"]


def clean_cds(ds):
    """Make a Copernicus file look like the WeatherBench2 one."""
    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    if "expver" in ds.dims:
        ds = ds.sel(expver=1, drop=True).combine_first(ds.sel(expver=5, drop=True))
    ds = ds.drop_vars([n for n in ["expver", "number"] if n in ds.variables])
    if "pressure_level" in ds.dims:
        ds = ds.rename({"pressure_level": "level"})
    return ds


class CopernicusQueue:
    """Downloads Copernicus data politely: only a few requests wait at a time."""

    DATASETS = {"sl": "reanalysis-era5-single-levels",
                "pl": "reanalysis-era5-pressure-levels"}

    def __init__(self, config, years=(2023, 2024, 2025, 2026),
                 months_per_request=6, max_active=2):
        self.cfg = config
        self.years = years
        self.max_active = {"sl": max_active, "pl": max_active}
        self.area = [37.5, 69.0, 7.5, 97.5]   # North, West, South, East
        self.ids_path = config.folder + "request_ids.json"
        self.ids = {}                           # file name -> request id
        if os.path.exists(self.ids_path):
            with open(self.ids_path) as f:
                self.ids = json.load(f)
        self.jobs = self._make_jobs(months_per_request)
        self.pause_until = 0
        key = getpass.getpass("Paste your CDS API key and press Enter: ")
        self.client = Client(url="https://cds.climate.copernicus.eu/api", key=key)

    # ---------- planning ----------
    def _make_jobs(self, months_per_request):
        """List every file we need: file name -> what to ask for."""
        jobs = {}
        for year in self.years:
            last_month = 10 if year == 2026 else 12
            months = list(range(1, last_month + 1))
            jobs[f"sl_{year}"] = dict(kind="sl", year=year, months=months)
            for i in range(0, len(months), months_per_request):
                part = months[i:i + months_per_request]
                name = f"pl_{year}_{part[0]:02d}_{part[-1]:02d}"
                jobs[name] = dict(kind="pl", year=year, months=part)
        return jobs

    def _split(self, name):
        """Replace one job by two jobs that each cover half of the months."""
        job = self.jobs.pop(name)
        half = len(job["months"]) // 2
        for part in (job["months"][:half], job["months"][half:]):
            new = f"{job['kind']}_{job['year']}_{part[0]:02d}_{part[-1]:02d}"
            self.jobs[new] = dict(kind=job["kind"], year=job["year"], months=part)

    def _path(self, name):
        return self.cfg.folder + name + ".nc"

    def _save_ids(self):
        with open(self.ids_path, "w") as f:
            json.dump(self.ids, f)

    def _request(self, job):
        request = dict(
            product_type=["reanalysis"],
            year=[str(job["year"])],
            month=[f"{m:02d}" for m in job["months"]],
            day=[f"{d:02d}" for d in range(1, 32)],
            time=["00:00", "06:00", "12:00", "18:00"],
            data_format="netcdf",
            download_format="unarchived",
            area=self.area,
            grid=[1.5, 1.5],
        )
        if job["kind"] == "sl":
            request["variable"] = SINGLE_VARS
        else:
            request["variable"] = PRESSURE_VARS
            request["pressure_level"] = ["500", "850"]
        return request

    # ---------- talking to Copernicus ----------
    def _submit(self, name):
        job = self.jobs[name]
        try:
            remote = self.client.submit(self.DATASETS[job["kind"]], self._request(job))
        except Exception as error:
            text = str(error)
            if ("cost limits" in text or "too large" in text) and len(job["months"]) > 1:
                print("request too large, splitting it in two:", name)
                self._split(name)
            else:
                print("could not submit", name, "- waiting 10 minutes:", text[:100])
                self.pause_until = time.time() + 600
            return
        self.ids[name] = remote.request_id
        self._save_ids()
        print("submitted", name)

    def submit_pending(self):
        """Send new requests, but keep only a few waiting per dataset."""
        if time.time() < self.pause_until:
            return
        for name in list(self.jobs):
            job = self.jobs.get(name)
            if job is None or name in self.ids or os.path.exists(self._path(name)):
                continue
            waiting = sum(1 for n in self.ids
                          if n in self.jobs and self.jobs[n]["kind"] == job["kind"]
                          and not os.path.exists(self._path(n)))
            if waiting >= self.max_active[job["kind"]]:
                continue
            self._submit(name)
            if time.time() < self.pause_until:
                return

    def check_requests(self):
        """Download finished requests. Drop requests that no longer exist."""
        for name, request_id in list(self.ids.items()):
            if os.path.exists(self._path(name)):
                continue
            try:
                remote = self.client.get_remote(request_id)
                status = remote.status
                if status == "successful":
                    remote.download(self._path(name))
                    print("downloaded", name)
                    continue
            except Exception as error:
                if "404" in str(error) or "Not Found" in str(error):
                    del self.ids[name]
                    self._save_ids()
                    print("old request", name, "no longer exists, removed from the list")
                else:
                    print("could not check", name, "-", str(error)[:80])
                continue
            if status not in ("accepted", "running"):
                kind = self.jobs.get(name, {}).get("kind", "pl")
                self.max_active[kind] = 1
                self.pause_until = time.time() + 600
                del self.ids[name]
                self._save_ids()
                print(f"{name} was rejected ({status}). Waiting 10 minutes, "
                      "then trying again with 1 request at a time.")

    def run(self):
        """Repeat every 2 minutes until every file is saved in Drive."""
        while True:
            self.submit_pending()
            self.check_requests()
            saved = sum(os.path.exists(self._path(n)) for n in self.jobs)
            waiting = sum(1 for n in self.ids if not os.path.exists(self._path(n)))
            print(f"{time.strftime('%H:%M')}  {saved} of {len(self.jobs)} files saved, "
                  f"{waiting} waiting at Copernicus")
            if saved == len(self.jobs):
                break
            time.sleep(120)

    # ---------- making the daily file ----------
    def _open_all(self, prefix):
        paths = []
        for year in self.years:
            paths += sorted(glob.glob(self.cfg.folder + f"{prefix}_{year}*.nc"))
        files = [clean_cds(xr.open_dataset(p)) for p in paths]
        return xr.concat(files, dim="time").sortby("time")

    def build_daily(self):
        sl = self._open_all("sl")
        pl = self._open_all("pl")
        daily = make_daily(
            t2m=sl["t2m"], u10=sl["u10"], v10=sl["v10"], msl=sl["msl"],
            z500=pl["z"].sel(level=500).drop_vars("level"),
            q850=pl["q"].sel(level=850).drop_vars("level"),
            t850=pl["t"].sel(level=850).drop_vars("level"),
        ).compute()
        daily.to_netcdf(self.cfg.folder + "india_daily_2023_2026.nc")
        return daily


queue = CopernicusQueue(cfg)
queue.run()
recent = queue.build_daily()
print(recent)

Paste your CDS API key and press Enter: ··········
submitted sl_2025
06:56  2 of 12 files saved, 4 waiting at Copernicus
old request sl_2025 no longer exists, removed from the list
06:59  2 of 12 files saved, 3 waiting at Copernicus
submitted sl_2025
07:01  2 of 12 files saved, 4 waiting at Copernicus


KeyboardInterrupt: 

For the missing year dataset

### 4.4 Retry-Based CDS Retrieval

A retry-based retrieval script is used to manage outstanding CDS requests when the server queue is full.

The script:
- Checks previously submitted requests.
- Downloads requests that have completed successfully.
- Submits pending requests when queue capacity permits.
- Saves request identifiers for continued monitoring.
- Stores downloaded files in Google Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ecmwf-datastores-client

Mounted at /content/drive


In [2]:
"""
retry_era5_requests.py

Keeps trying to send ERA5 requests to Copernicus while its queue is full.
Only 2 requests wait at a time, and every finished file is saved to Drive.
Only the hot season (Feb to Jun) is requested, so each request is small.
"""
import getpass
import json
import os
import time

from ecmwf.datastores import Client

URL = "https://cds.climate.copernicus.eu/api"
AREA = [37.5, 69.0, 7.5, 97.5]          # North, West, South, East (India)
FOLDER = "/content/drive/MyDrive/heatwave_project/"
WAIT_SECONDS = 120                      # pause between rounds
MAX_WAITING = 2                         # requests allowed to wait at one time (in total)

DATASETS = {"sl": "reanalysis-era5-single-levels",
            "pl": "reanalysis-era5-pressure-levels"}
VARIABLES = {"sl": ["2m_temperature", "10m_u_component_of_wind",
                    "10m_v_component_of_wind", "mean_sea_level_pressure"],
             "pl": ["geopotential", "specific_humidity", "temperature"]}

# Most important first: (name, kind, year, first month, last month)
JOBS = [
    ("pl_2024_02_06", "pl", 2024, 2, 6),
    ("sl_2025_02_06", "sl", 2025, 2, 6),
    ("pl_2025_02_06", "pl", 2025, 2, 6),
    ("sl_2026_02_06", "sl", 2026, 2, 6),
    ("pl_2026_02_06", "pl", 2026, 2, 6),
]
KIND = {name: kind for name, kind, *_ in JOBS}
IDS_FILE = FOLDER + "retry_request_ids.json"


def build_request(kind, year, first_month, last_month):
    request = {
        "product_type": ["reanalysis"],
        "variable": VARIABLES[kind],
        "year": [str(year)],
        "month": [f"{m:02d}" for m in range(first_month, last_month + 1)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": ["00:00", "06:00", "12:00", "18:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": AREA,
        "grid": [1.5, 1.5],
    }
    if kind == "pl":
        request["pressure_level"] = ["500", "850"]
    return request


def path_of(name):
    return FOLDER + name + ".nc"


def load_ids():
    if os.path.exists(IDS_FILE):
        with open(IDS_FILE) as f:
            return json.load(f)
    return {}


def save_ids(ids):
    with open(IDS_FILE, "w") as f:
        json.dump(ids, f, indent=2)


def collect_finished(client, ids):
    """Download requests that are done; forget requests that failed or vanished."""
    for name, request_id in list(ids.items()):
        if os.path.exists(path_of(name)):
            continue
        try:
            remote = client.get_remote(request_id)
            if remote.status == "successful":
                remote.download(path_of(name))
                print("downloaded", name)
            elif remote.status not in ("accepted", "running"):
                raise RuntimeError("status " + remote.status)
        except Exception as error:
            print("dropping", name, "- will send it again:", str(error)[:80])
            del ids[name]
            save_ids(ids)


def send_new(client, ids):
    """Send new requests while there is room. Returns the datasets whose queue is full."""
    queue_full = set()
    for name, kind, year, first_month, last_month in JOBS:
        if name in ids or os.path.exists(path_of(name)) or kind in queue_full:
            continue
        waiting = sum(1 for n in ids if not os.path.exists(path_of(n)))
        if waiting >= MAX_WAITING:
            break
        try:
            remote = client.submit(DATASETS[kind],
                                   build_request(kind, year, first_month, last_month))
        except Exception as error:
            text = str(error)
            if "queued requests" in text or "Maximum number" in text:
                queue_full.add(kind)
            else:
                raise
            continue
        ids[name] = remote.request_id
        save_ids(ids)
        print("submitted", name)
    return queue_full


def main():
    key = getpass.getpass("Paste your CDS API key and press Enter: ")
    client = Client(url=URL, key=key)
    ids = load_ids()
    while True:
        collect_finished(client, ids)
        queue_full = send_new(client, ids)
        saved = sum(os.path.exists(path_of(name)) for name, *_ in JOBS)
        print(f"{time.strftime('%H:%M')}  {saved} of {len(JOBS)} files saved, "
              f"{len(ids) - saved} waiting, queue full for: {sorted(queue_full) or 'none'}")
        if saved == len(JOBS):
            break
        time.sleep(WAIT_SECONDS)
    print("All files are in your Drive folder heatwave_project.")


main()

Paste your CDS API key and press Enter: ··········


526f50f62a8536362716c570abd66699.nc:   0%|          | 0.00/3.00M [00:00<?, ?B/s]

downloaded pl_2026_02_06
13:59  5 of 5 files saved, 0 waiting, queue full for: none
All files are in your Drive folder heatwave_project.



## 5. Downloaded Dataset Inspection

Before merging, the downloaded NetCDF files are inspected to verify their availability and structure.

The inspection includes:

- File names and file sizes.
- Available variables.
- Time coordinate information.
- Starting and ending dates.
- Number of available time steps.

This helps identify missing files and verify the downloaded dataset periods before merging.


In [7]:
import glob, os
import xarray as xr

def inspect_files(folder):
    """Print the variables and date range of every .nc file in the folder."""
    for path in sorted(glob.glob(folder + "*.nc")):
        name = os.path.basename(path)
        size = os.path.getsize(path) / 1e6
        try:
            ds = xr.open_dataset(path)
            time_name = "valid_time" if "valid_time" in ds.coords else "time"
            t = ds[time_name].values
            print(f"{name:45s} {size:7.1f} MB | {time_name}: {str(t[0])[:10]} to {str(t[-1])[:10]} ({len(t)} steps)")
            print("      variables:", list(ds.data_vars))
            ds.close()
        except Exception as e:
            print(f"{name:45s} could not be opened: {e}")

inspect_files(cfg.folder)

2988ff4002fd6bafe0fad6ba5960f7f3.nc               1.9 MB | valid_time: 2023-01-01 to 2023-03-31 (360 steps)
      variables: ['z', 'q', 't']
5255f0f38c3837b8dd8e87d3966e6a7a.nc               2.4 MB | valid_time: 2025-02-01 to 2025-06-30 (600 steps)
      variables: ['t2m', 'u10', 'v10', 'msl']
59dd5ab670d8d088fa0d358a22ff84f0.nc               5.7 MB | valid_time: 2024-01-01 to 2024-12-31 (1464 steps)
      variables: ['t2m', 'u10', 'v10', 'msl']
671a30cbbb35d4194fc2bbae45849ed1.nc               3.2 MB | valid_time: 2024-02-01 to 2024-06-30 (604 steps)
      variables: ['z', 'q', 't']
c682ed3e5e04811d6af1e56aca959a5c.nc               5.7 MB | valid_time: 2023-01-01 to 2023-12-31 (1460 steps)
      variables: ['t2m', 'u10', 'v10', 'msl']
hist_1990.nc                                      4.9 MB | time: 1990-01-01 to 1990-12-31 (365 steps)
      variables: ['tmax', 'tmean', 'u10', 'v10', 'msl', 'z500', 'q850', 't850']
hist_1991.nc                                      4.9 MB | time: 1991-01


## 6. Dataset Merging and Daily Data Preparation

The historical WeatherBench 2 dataset and additional CDS datasets are combined to create a unified daily dataset.

The merging procedure:

1. Opens the historical and recent NetCDF files.
2. Standardises the time coordinate.
3. Generates daily values from six-hourly observations.
4. Aligns latitude and longitude coordinates.
5. Compares overlapping January 2023 observations to check consistency between data sources.
6. Combines the datasets chronologically.
7. Checks for duplicate dates and missing values.
8. Saves the final merged dataset in NetCDF format.

Output file: `india_daily_full.nc`


In [9]:
import numpy as np
import xarray as xr

# (single-level file, pressure-level file) for each hot season
SEASON_FILES = {
    2023: ("sl_2023.nc", "pl_2023_01_06.nc"),
    2024: ("59dd5ab670d8d088fa0d358a22ff84f0.nc", "pl_2024_02_06.nc"),
    2025: ("sl_2025_02_06.nc", "pl_2025_02_06.nc"),
    2026: ("sl_2026_02_06.nc", "pl_2026_02_06.nc"),
}


def open_cds(path):
    """Open a Copernicus file and give its time axis the name 'time'."""
    ds = xr.open_dataset(path)
    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    return ds.drop_vars([c for c in ("expver", "number") if c in ds.coords])


def build_daily(sl_path, pl_path):
    """Turn 6-hourly single-level + pressure-level files into daily values."""
    sl, pl = open_cds(sl_path), open_cds(pl_path)
    sl = sl.sel(time=slice(pl.time.values[0], pl.time.values[-1]))   # same period
    day = lambda x, how: getattr(x.resample(time="1D"), how)()
    return xr.Dataset({
        "tmax":  day(sl.t2m, "max"),
        "tmean": day(sl.t2m, "mean"),
        "u10":   day(sl.u10, "mean"),
        "v10":   day(sl.v10, "mean"),
        "msl":   day(sl.msl, "mean"),
        "z500":  day(pl.z.sel(pressure_level=500, drop=True), "mean"),
        "q850":  day(pl.q.sel(pressure_level=850, drop=True), "mean"),
        "t850":  day(pl.t.sel(pressure_level=850, drop=True), "mean"),
    })


def check_against_old(new, old):
    """Compare 1-10 Jan 2023 with the old file; fix a kelvin offset if found."""
    days = slice("2023-01-01", "2023-01-10")
    a = new.sel(time=days)
    b = old.sel(time=days).reindex(latitude=a.latitude, longitude=a.longitude, method="nearest")
    print("Variable | mean(new-old) | mean|new-old| | old std")
    for v in old.data_vars:
        d = float((a[v] - b[v]).mean())
        if abs(d - 273.15) < 5:                    # new file is in kelvin, old in Celsius
            new[v] = new[v] - 273.15
            a = new.sel(time=days)
            d = float((a[v] - b[v]).mean())
            print(f"  ({v}: converted kelvin to Celsius)")
        print(f"{v:6s} | {d:12.3f} | {float(abs(a[v] - b[v]).mean()):12.3f} | {float(b[v].std()):.3f}")
    return new


def merge_all(config):
    old = xr.open_dataset(config.folder + "india_daily_1990_2023.nc")
    parts = []
    for year, (sl_name, pl_name) in SEASON_FILES.items():
        daily = build_daily(config.folder + sl_name, config.folder + pl_name)
        daily = daily.reindex(latitude=old.latitude, longitude=old.longitude, method="nearest")
        if year == 2023:
            daily = check_against_old(daily, old)
            daily = daily.sel(time=slice("2023-01-11", None))     # old file already has 1-10 Jan
        print(year, "days added:", daily.sizes["time"],
              "| first:", str(daily.time.values[0])[:10], "| last:", str(daily.time.values[-1])[:10])
        parts.append(daily)

    full = xr.concat([old] + parts, dim="time").sortby("time")
    print("Duplicate days:", full.sizes["time"] - len(np.unique(full.time.values)))
    print("Missing values (NaN):", int(full.to_array().isnull().sum()))
    print("Total days:", full.sizes["time"], "| grid:", full.sizes["latitude"], "x", full.sizes["longitude"])
    full.to_netcdf(config.folder + "india_daily_full.nc")
    print("Saved india_daily_full.nc")
    return full


full = merge_all(cfg)

Variable | mean(new-old) | mean|new-old| | old std
  (tmax: converted kelvin to Celsius)
tmax   |        0.127 |        0.871 | 15.154
  (tmean: converted kelvin to Celsius)
tmean  |        0.038 |        0.754 | 16.644
u10    |       -0.016 |        0.268 | 2.192
v10    |        0.021 |        0.276 | 2.221
msl    |       -2.286 |       34.601 | 642.074
z500   |        0.432 |        3.359 | 742.332
q850   |       -0.000 |        0.000 | 0.004
  (t850: converted kelvin to Celsius)
t850   |        0.013 |        0.244 | 4.681
2023 days added: 171 | first: 2023-01-11 | last: 2023-06-30
2024 days added: 151 | first: 2024-02-01 | last: 2024-06-30
2025 days added: 150 | first: 2025-02-01 | last: 2025-06-30
2026 days added: 150 | first: 2026-02-01 | last: 2026-06-30
Duplicate days: 0
Missing values (NaN): 0
Total days: 12685 | grid: 21 x 20
Saved india_daily_full.nc
